# 08 · A complete training loop with DataLoaders

**Objective:** To build, train, and evaluate a complete, simple AI model for a regression task using a standard, scalable workflow.

## Part 1: Concepts (90 mins)

### 1. The ML Workflow Revisited
We will now enhance the full-batch training workflow of notebook 06 by incorporating the `Dataset` and `DataLoader` of notebook 07. This is how modern PyTorch projects are structured.

The cycle remains the same, but now we iterate through batches of data instead of the whole dataset at once.

1.  **Prepare Data**: Wrap data in a `Dataset` and a `DataLoader`.
2.  **Loop through Epochs**: An epoch is one full pass over the entire dataset.
3.  **Loop through Batches**: Inside each epoch, loop through the batches provided by the `DataLoader`.
4.  **For each batch**:
    a. **Forward Pass:** Get the model's prediction.
    b. **Compute Loss:** Compare prediction with the true values.
    c. **Backward Pass:** Calculate gradients.
    d. **Update Weights:** Use the optimizer to update model parameters.

### 2. Building Models with `torch.nn.Module` (Review)
We define our model architecture by creating a class that inherits from `nn.Module`.

In [ ]:
import torch
import torch.nn as nn

class LinearRegression(nn.Module):
    def __init__(self):
        super(LinearRegression, self).__init__()
        self.linear = nn.Linear(1, 1) # 1 input feature, 1 output feature

    def forward(self, x):
        return self.linear(x)

### 3. Loss Functions and Optimizers (Review)
- **Loss Function:** Measures how wrong the model is. `nn.MSELoss` for regression.
- **Optimizer:** Updates model weights to reduce the loss. `torch.optim.SGD` is a simple and effective choice.

### 4. The Training Loop with Mini-batches
This is the core of the training process, now updated to use a `DataLoader`:

```python
for epoch in range(epochs):
    for batch_X, batch_y in data_loader:
        y_pred = model(batch_X)          # a. forward pass
        loss = loss_fn(y_pred, batch_y)  # b. compute loss
        optimizer.zero_grad()            #    reset gradients from the previous step
        loss.backward()                  # c. backward pass
        optimizer.step()                 # d. update weights
```

## Part 2: Exercises & Debugging (90 mins) - Guided Coding

### Lab 8.1–8.4: Full Model Training with DataLoader
* **Task:** Combine all concepts into a single script. Create synthetic data, wrap it in a `Dataset` and `DataLoader`, build the model, write the full training loop, and evaluate the result.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(0)  # reproducible data, initialization and shuffling

# --- 1. Data Preparation ---
true_weight = 3
true_bias = 5
X = torch.arange(0, 1, 0.01).unsqueeze(1)
y = true_weight * X + true_bias + torch.randn(X.shape) * 0.1

# --- 2. Create Custom Dataset and DataLoader (Lab 8.1) ---
class MyRegressionDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = MyRegressionDataset(X, y)
data_loader = DataLoader(dataset=dataset, batch_size=10, shuffle=True)

# --- 3. Model, Loss, Optimizer (Lab 8.2) ---
class LinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(in_features=1, out_features=1)
    def forward(self, x):
        return self.linear(x)

model = LinearRegression()
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# --- 4. The Training Loop (Lab 8.3) ---
epochs = 100
for epoch in range(epochs):
    for batch_X, batch_y in data_loader:
        # Forward pass
        y_pred = model(batch_X)
        # Loss calculation
        loss = loss_fn(y_pred, batch_y)
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        # Weight update
        optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# --- 5. Make Predictions (Lab 8.4) ---
model.eval()
with torch.no_grad():
    new_x = torch.tensor([[0.5]])
    prediction = model(new_x)
    print(f"\nPrediction for x=0.5: {prediction.item():.4f}")
    # True value is 3 * 0.5 + 5 = 6.5
    
    # Visualize the result
    predicted_line = model(X).detach().numpy()
    plt.figure(figsize=(8, 6))
    plt.scatter(X.numpy(), y.numpy(), label='Original data', alpha=0.6)
    plt.plot(X.numpy(), predicted_line, color='r', label='Fitted line')
    plt.title("Linear Regression Fit")
    plt.xlabel("X")
    plt.ylabel("y")
    plt.legend()
    plt.show()

### Debugging Focus: Learning Rate Tuning
- **The Problem:** Your loss is either exploding (going to `NaN`) or not decreasing at all. This is the most common training issue.
- **The Cause:** The **learning rate (`lr`)** in your optimizer is the single most important hyperparameter to tune.
- **How to Fix:**
    - **If loss explodes:** Your `lr` is too high. The optimizer is overshooting the minimum. **Decrease it by a factor of 10** (e.g., from `0.1` to `0.01`).
    - **If loss is stagnant:** Your `lr` is too low. The steps are too small to make meaningful progress. **Increase it by a factor of 10** (e.g., from `0.001` to `0.01`).
    - Finding the right learning rate is an empirical process of trial and error.